READING THE DATA SET

In [ ]:
import pandas as pd
df = pd.read_csv('C:/Users/Administrator/Documents/Dhruvil/Projects/Data Science/Data Science xl files/neo.csv')
print(df,'\n\n',df.info,df.dtypes)


HANDLING MISSING VALUES USING MEAN/MEDIAN/MODE

In [ ]:
print(df[df.isnull().any(axis=1)])
df = df.fillna({'est_diameter_min': df['est_diameter_min'].mean(), 'est_diameter_max': df['est_diameter_max'].mean(), 'relative_velocity': df['relative_velocity'].mean(), 'miss_distance' : df['miss_distance'].mean(), 'absolute_magnitude' : df['absolute_magnitude'].mean()})
print(df)


CONVERTING CATEGORICAL FEATURES INTO NUMERICAL USING ENCODING

In [ ]:
df['hazardous'] = df['hazardous'].astype(int)
print(df)

NORMALIZE/STANDARDIZE THE NUMERICAL FEATURES

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_features = [
    'est_diameter_min', 
    'est_diameter_max', 
    'relative_velocity', 
    'miss_distance', 
    'absolute_magnitude'
]

# Apply StandardScaler
scaler_std = StandardScaler()
df_standardized = df.copy()
df_standardized[numerical_features] = scaler_std.fit_transform(df[numerical_features])

# Print summary to verify mean is ~0 and std is ~1
print(df_standardized[numerical_features].describe().loc[['mean', 'std']])
print('\n\n',df)

VISUALIZING OUTLIERS USING BOXPLOTS AND REMOVE THEM

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the dataset and select continuous numerical features

numerical_features = [
    'est_diameter_min', 
    'est_diameter_max', 
    'relative_velocity', 
    'miss_distance', 
    'absolute_magnitude'
]

# 2. Visualize Outliers BEFORE Removal
plt.figure(figsize=(12, 6))

for i, col in enumerate(numerical_features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(f"{col} (Original)")
    plt.ylabel('')
plt.tight_layout()
plt.show()

# 3. Remove Outliers using the Interquartile Range (IQR) Method
df_clean = df.copy()
for col in numerical_features:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # Define standard bounds (1.5 * IQR rule)
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Keep only rows within the valid bounds
    df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

# Print summary statistics of the removal process
print(f"Original dataset rows: {len(df)}")
print(f"Cleaned dataset rows:  {len(df_clean)}")
print(f"Total rows removed:    {len(df) - len(df_clean)}")

# 4. Visualize AFTER Removal to verify the cleaned distribution
plt.figure(figsize=(12, 6))
for i, col in enumerate(numerical_features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df_clean[col], color='lightgreen')
    plt.title(f"{col} (Cleaned)")
    plt.ylabel('')
plt.tight_layout()
plt.show()